# Feature Selection and Hyperparameter Tunning across all cell types

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
import sys
import warnings

project_dir = Path.cwd().parent

sys.path.insert(0, str(Path.cwd().parent)) # import src/
warnings.filterwarnings("ignore")

pd.set_option("display.width", 220, "display.max_columns", 30)

from src import (NestedCVRegressor, default_models,
                           default_param_spaces, MRMRSelector, run_per_family,
                           alt_param_spaces)

HVG_DATA_DIR = project_dir / 'data' / 'HVGs'

## Dataset

In [2]:
FAMILIES = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells_train_hvg.h5ad",  HVG_DATA_DIR / "Kenyon_cells_test_hng.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe_train_hvg.h5ad",    HVG_DATA_DIR / "Optic_lobe_test_hng.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic_train_hvg.h5ad",HVG_DATA_DIR / "Monoaminergic_test_hng.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia_train_hvg.h5ad",         HVG_DATA_DIR / "Glia_test_hng.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic_train_hvg.h5ad",  HVG_DATA_DIR / "Peptidergic_test_hng.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock_train_hvg.h5ad",        HVG_DATA_DIR / "Clock_test_hng.h5ad"),
}
FAMILIES = {f: p for f, p in FAMILIES.items() if p[0].exists() and p[1].exists()}

## Baseline over all cell types

In [ ]:
fs_tune_baseline = run_per_family(
    FAMILIES,
    models=default_models(),
    param_spaces=alt_param_spaces(),
    feature_selector=MRMRSelector(k=100, relevance="f"),
    tune=True,
    dev="train",
    n_rounds=5,
    n_outer=5,
    n_inner=3,
    n_trials=20,
    inner_scoring="neg_root_mean_squared_error",
    keep_models=False,
    save_dir="../results/fs_tune_baseline",
    save_name="fs_tune_baseline",
    verbose=1,
)
NestedCVRegressor.format_table(fs_tune_baseline.summary)


=== Family: KenyonCells  (2278 cells, 2001 features) ===
[KenyonCells] Dummy: baseline ...
[KenyonCells] LinearRegression: baseline ...
[KenyonCells] ElasticNet: tuning ...
[KenyonCells] SVR: tuning ...
[KenyonCells] RandomForest: tuning ...
[KenyonCells] RandomForest_authors: baseline ...
[KenyonCells] XGBoost: tuning ...

=== Family: OpticLobe  (6365 cells, 2001 features) ===
[OpticLobe] Dummy: baseline ...
[OpticLobe] LinearRegression: baseline ...
[OpticLobe] ElasticNet: tuning ...
[OpticLobe] SVR: tuning ...
[OpticLobe] RandomForest: tuning ...
[OpticLobe] RandomForest_authors: baseline ...
[OpticLobe] XGBoost: tuning ...

=== Family: Monoaminergic  (748 cells, 2001 features) ===
[Monoaminergic] mRMR K: 100 -> 79 (size-aware cap)
[Monoaminergic] Dummy: baseline ...
[Monoaminergic] LinearRegression: baseline ...
[Monoaminergic] ElasticNet: tuning ...
[Monoaminergic] SVR: tuning ...
[Monoaminergic] RandomForest: tuning ...
[Monoaminergic] RandomForest_authors: baseline ...
[Monoami

,model,MAE,RMSE,R2,Spearman
0,Dummy,6.304 (6.251–6.312),9.908 (9.713–9.911),-0.000 (-0.000–-0.000),nan (nan–nan)
1,LinearRegression,4.232 (4.166–4.338),6.320 (6.174–6.482),0.588 (0.572–0.605),0.707 (0.689–0.720)
2,ElasticNet,4.129 (4.059–4.233),6.229 (6.101–6.391),0.598 (0.580–0.615),0.718 (0.705–0.734)
3,SVR,3.440 (3.391–3.542),5.726 (5.661–5.800),0.658 (0.641–0.671),0.773 (0.763–0.786)
4,RandomForest,2.980 (2.949–3.007),5.552 (5.321–5.769),0.685 (0.654–0.704),0.836 (0.826–0.845)
5,RandomForest_authors,4.166 (4.078–4.239),6.660 (6.484–6.997),0.535 (0.483–0.572),0.622 (0.584–0.650)
6,XGBoost,2.920 (2.870–2.969),5.183 (5.102–5.337),0.721 (0.698–0.734),0.833 (0.812–0.844)
7,Dummy,6.924 (6.913–6.936),10.621 (10.602–10.670),-0.000 (-0.000–-0.000),nan (nan–nan)
8,LinearRegression,4.177 (4.113–4.214),6.188 (6.032–6.236),0.664 (0.654–0.679),0.731 (0.724–0.738)
9,ElasticNet,4.102 (4.055–4.142),6.167 (6.030–6.229),0.666 (0.655–0.678),0.745 (0.732–0.749)


## Best model per cell type

In [11]:
# best model per cell type by median MAE
best_per_type_mae = fs_tune_baseline.long.groupby(["family", "model"])["MAE"].median().reset_index()
best_per_type_mae = best_per_type_mae.loc[best_per_type_mae.groupby("family")["MAE"].idxmin()].sort_values("MAE")
display(best_per_type_mae)

,family,model,MAE
31,OpticLobe,RandomForest,2.643220
20,KenyonCells,XGBoost,2.920422
38,Peptidergic,RandomForest,4.427933
24,Monoaminergic,RandomForest,4.629291
6,Clock,XGBoost,5.405836
13,Glia,XGBoost,5.841202


In [15]:
# best model per cell type by median RMSE
best_per_type_rmse = fs_tune_baseline.long.groupby(["family", "model"])[["RMSE", "MAE"]].median().reset_index()
best_per_type_rmse = best_per_type_rmse.loc[best_per_type_rmse.groupby("family")["RMSE"].idxmin()].sort_values("RMSE")
display(best_per_type_rmse)

,family,model,RMSE,MAE
34,OpticLobe,XGBoost,4.693271,2.647844
20,KenyonCells,XGBoost,5.182812,2.920422
41,Peptidergic,XGBoost,7.128842,4.492175
27,Monoaminergic,XGBoost,7.368947,4.661847
13,Glia,XGBoost,8.251753,5.841202
6,Clock,XGBoost,8.574659,5.405836


In [ ]:
# best model per cell type by median RMSE, with MAE, R2, Pearson, Spearman (all with CIs)
best_per_type_rmse_ci = (
    fs_tune_baseline.summary
    .loc[fs_tune_baseline.summary.groupby("family")["RMSE_median"].idxmin()]
    .sort_values("RMSE_median")
    [["family", "model",
      "RMSE_median", "RMSE_lo", "RMSE_hi",
      "MAE_median",  "MAE_lo",  "MAE_hi",
      "R2_median",   "R2_lo",   "R2_hi",
      "Pearson_median", "Pearson_lo", "Pearson_hi",
      "Spearman_median", "Spearman_lo", "Spearman_hi"]]
)
# display(best_per_type_rmse_ci)

disp = best_per_type_rmse_ci.copy()
for m in ["RMSE", "MAE", "R2", "Pearson", "Spearman"]:
    disp[m] = disp.apply(
        lambda r: f"{r[f'{m}_median']:.3f} ({r[f'{m}_lo']:.3f}-{r[f'{m}_hi']:.3f})", axis=1)
disp = disp[["family", "model", "RMSE", "MAE", "R2", "Pearson", "Spearman"]]
display(disp)

,family,model,RMSE,MAE,R2,Pearson,Spearman
13,OpticLobe,XGBoost,4.693 (4.640-4.824),2.648 (2.593-2.675),0.805 (0.793-0.809),0.898 (0.892-0.901),0.878 (0.871-0.885)
6,KenyonCells,XGBoost,5.183 (5.102-5.337),2.920 (2.870-2.969),0.721 (0.698-0.734),0.852 (0.836-0.864),0.833 (0.812-0.844)
34,Peptidergic,XGBoost,7.129 (6.970-7.390),4.492 (4.406-4.575),0.675 (0.653-0.691),0.828 (0.817-0.836),0.804 (0.791-0.823)
20,Monoaminergic,XGBoost,7.369 (7.104-7.649),4.662 (4.479-4.798),0.652 (0.643-0.690),0.825 (0.807-0.845),0.781 (0.770-0.793)
27,Glia,XGBoost,8.252 (8.182-8.301),5.841 (5.762-5.884),0.817 (0.814-0.820),0.905 (0.904-0.908),0.902 (0.900-0.908)
41,Clock,XGBoost,8.575 (8.286-9.261),5.406 (5.227-5.770),0.569 (0.495-0.597),0.770 (0.727-0.783),0.770 (0.752-0.800)


## Results examples

In [6]:
fs_tune_baseline.results.keys()

dict_keys(['KenyonCells', 'OpticLobe', 'Monoaminergic', 'Glia', 'Peptidergic', 'Clock'])

In [7]:
fs_tune_baseline.results['Clock'].summary

,model,MAE_median,MAE_lo,MAE_hi,RMSE_median,RMSE_lo,RMSE_hi,MedAE_median,MedAE_lo,MedAE_hi,R2_median,R2_lo,R2_hi,Pearson_median,Pearson_lo,Pearson_hi,Spearman_median,Spearman_lo,Spearman_hi
0,Dummy,9.727725,9.691544,9.758684,12.990611,12.981979,13.054431,8.753799,8.742424,8.790274,-0.000108,-0.000358,-0.000064,NaN,NaN,NaN,NaN,NaN,NaN
1,LinearRegression,7.424692,7.183383,7.893197,11.468839,10.640280,12.246578,4.844310,4.600759,5.368298,0.220516,0.110055,0.363493,0.591450,0.532013,0.632801,0.673139,0.643434,0.708051
2,ElasticNet,7.266105,6.806249,7.487565,10.659566,10.056338,11.307629,4.685008,4.167596,5.115302,0.332750,0.228428,0.401531,0.600213,0.562391,0.641314,0.696531,0.672164,0.731399
3,SVR,7.209404,6.742077,7.566762,10.465761,9.891702,10.968384,4.006250,3.695252,4.432676,0.401991,0.295496,0.423795,0.634153,0.572794,0.653694,0.653631,0.627092,0.714436
4,RandomForest,5.499584,5.076943,5.751327,8.796668,8.103333,9.268309,2.944010,2.729933,3.177690,0.547440,0.496569,0.603398,0.763242,0.721213,0.804516,0.785324,0.768988,0.811356
5,RandomForest_authors,6.547339,6.293611,6.748895,9.853119,9.650523,10.349336,4.040189,3.873703,4.221860,0.416623,0.390181,0.476187,0.677409,0.637853,0.736243,0.658081,0.616784,0.690469
6,XGBoost,5.405836,5.226565,5.769825,8.574659,8.285657,9.261369,2.977393,2.772210,3.256146,0.568516,0.494997,0.597050,0.770020,0.726723,0.782902,0.770094,0.751556,0.800265
